In [ ]:
# %%bash
# pip uninstall basicsr -y
# rm -rf basicsr.egg-info build dist
# find . -name "__pycache__" -type d -exec rm -r {} +

Found existing installation: basicsr 1.3.4.6
Uninstalling basicsr-1.3.4.6:
  Successfully uninstalled basicsr-1.3.4.6


In [ ]:
!pip install --upgrade pip setuptools wheel modelscope
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [3]:
!pip install lpips opencv-python scikit-image numpy 

In [ ]:
!pip install -r requirements312.txt

In [ ]:
import torch
torch.__version__
!git config --global --add safe.directory /home/h/DDColor

In [3]:
!pip install -e . --no-build-isolation
#!python setup.py develop

Obtaining file:///home/h/DDColor
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basicsr (pyproject.toml) ... done
  Created wheel for basicsr: filename=basicsr-1.3.4.6-0.editable-py3-none-any.whl size=10285 sha256=b5dbf79b090b469b795c2d5b6c138f056a7b0170363664f9557123d337323d39
  Stored in directory: /tmp/pip-ephem-wheel-cache-88htgw0z/wheels/53/2d/35/2387e1dfa7d036b42a753336f85b3561c959049bc6cf833672
Successfully built basicsr


In [ ]:
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download('damo/cv_ddcolor_image-colorization', cache_dir='./modelscope')
print('model assets saved to %s' % model_dir)

In [3]:
# chuan bi data
import os
import pandas as pd
import shutil
from tqdm import tqdm

def prepare_data(csv_file, split_name):
    # Tên folder gốc chứa ảnh của bạn
    base_dir = 'ViCoW_Dataset' 
    
    if not os.path.exists(csv_file):
        print(f"Bỏ qua: Không tìm thấy file {csv_file}")
        return

    # Tạo thư mục đích: dataset/train, dataset/val, dataset/test
    output_dir = os.path.join('dataset', split_name)
    os.makedirs(output_dir, exist_ok=True)

    # Đọc CSV
    df = pd.read_csv(csv_file)
    print(f"\n--- Đang xử lý tập {split_name.upper()} ({len(df)} ảnh) ---")
    
    success_count = 0
    missing_samples = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Đường dẫn từ CSV: Color_image/VIDEO2_VaoNamRaBac/frame_0998.jpg
        rel_path = row['colorPath']
        
        # Ghép thành đường dẫn thực tế: ViCoW_Dataset/Color_image/...
        src_path = os.path.join(base_dir, rel_path)
        
        if os.path.exists(src_path):
            # Tách lấy tên VIDEO và tên FRAME để tạo tên file mới (tránh trùng)
            # Ví dụ: VIDEO2_VaoNamRaBac_frame_0998.jpg
            path_parts = rel_path.split('/')
            video_folder = path_parts[1] 
            filename = path_parts[-1]
            new_filename = f"{video_folder}_{filename}"
            
            dst_path = os.path.join(output_dir, new_filename)
            shutil.copy2(src_path, dst_path)
            success_count += 1
        else:
            if len(missing_samples) < 1: # Lưu lại mẫu lỗi đầu tiên để báo cáo
                missing_samples.append(os.path.abspath(src_path))

    print(f"Kết quả: Copy thành công {success_count}/{len(df)} ảnh vào '{output_dir}'")
    
    if success_count == 0 and len(missing_samples) > 0:
        print(f"\n[CẢNH BÁO] Không tìm thấy ảnh nào! Script đã thử tìm ở:")
        print(f" -> {missing_samples[0]}")
        print("Hãy đảm bảo folder 'ViCoW_Dataset' nằm cùng cấp với file script này.")

# --- Chạy script ---
tasks = [('ViCoW_Dataset/train.csv', 'train'), ('ViCoW_Dataset/val.csv', 'val'), ('ViCoW_Dataset/test.csv', 'test')]
for csv, split in tasks:
    prepare_data(csv, split)

print("\nHoàn tất!")


--- Đang xử lý tập TRAIN (1327 ảnh) ---


100%|██████████| 1327/1327 [00:01<00:00, 665.50it/s]


Kết quả: Copy thành công 1327/1327 ảnh vào 'dataset/train'

--- Đang xử lý tập VAL (187 ảnh) ---


100%|██████████| 187/187 [00:00<00:00, 635.71it/s]


Kết quả: Copy thành công 187/187 ảnh vào 'dataset/val'

--- Đang xử lý tập TEST (382 ảnh) ---


100%|██████████| 382/382 [00:00<00:00, 616.32it/s]

Kết quả: Copy thành công 382/382 ảnh vào 'dataset/test'

Hoàn tất!


In [7]:
%%bash
python data_list/get_meta_file.py --output-name ./dataset/train.txt --data-path ./dataset/train
python data_list/get_meta_file.py --output-name ./dataset/val.txt --data-path ./dataset/val
python data_list/get_meta_file.py --output-name ./dataset/test.txt --data-path ./dataset/test

Generating ./dataset/train.txt from ./dataset/train ...


100%|██████████| 1327/1327 [00:00<00:00, 2586357.53it/s]


Done.
Generating ./dataset/val.txt from ./dataset/val ...


100%|██████████| 187/187 [00:00<00:00, 1415766.87it/s]


Done.
Generating ./dataset/test.txt from ./dataset/test ...


100%|██████████| 382/382 [00:00<00:00, 2083516.42it/s]


Done.


In [ ]:
!wget https://dl.fbaipublicfiles.com/convnext/convnext_large_22k_224.pth -P pretrain/
!wget https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth -P pretrain/

In [8]:
# /home/h/DDColor/options/train/train_ViCoW.yml
!sh scripts/train.sh

/home/h/ddcolor312/lib/python3.12/site-packages/torch/distributed/launch.py:208: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  main()
Path already exists. Rename it to /home/h/DDColor/experiments/train_ViCoW_1.0_0.5_freeze_g_pretrain_512_archived_20260103_204516
Path already exists. Rename it to /home/h/DDColor/tb_logger/train_ViCoW_1.0_0.5_freeze_g_pretrain_512_archived_20260103_204516
2026-01-03 20:45:16,814 INFO: 
                ____                _       _____  ____
               / __ ) ____ _ _____ (_)_____/ ___/ / __ \
              / __  |/ __ `// ___// // ___/\__ \ / /_/ /
             / /_/ // /_/ /(__  )/ // /__ ___/ // _, _/
            /___

### Tensorboard

In [ ]:
!tensorboard --logdir=tb_logger/train_ViCoW_1.2_0.5_freeze_g_pretrain --port=6006

/home/h/ddcolor312/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.20.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C


## Infer, gen out_test

In [ ]:
# !python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ./assets/test_images

In [5]:
#!python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512 --model_size large
#DDColor/experiments/train_ViCoW/models/net_g_5000.pth

!python infer.py --model_path experiments/train_ViCoW_1.0_0.5_freeze_g_pretrain/models/net_g_20000.pth \
                --input dataset/test   \
                --output out_train_ViCoW_1.0_0.5_freeze_g_pretrain_20000 \
                --input_size 512 # --model_size large

!python infer.py --model_path experiments/train_ViCoW_1.0_0.5_freeze_g_pretrain/models/net_g_10000.pth \
                --input dataset/test   \
                --output out_train_ViCoW_1.0_0.5_freeze_g_pretrain_10000 \
                --input_size 512 # --model_size large





# from infer_hf import DDColorHF

# ddcolor_paper_tiny = DDColorHF.from_pretrained("piddnad/ddcolor_paper_tiny")
# ddcolor_paper      = DDColorHF.from_pretrained("piddnad/ddcolor_paper")
# ddcolor_modelscope = DDColorHF.from_pretrained("piddnad/ddcolor_modelscope")
# ddcolor_artistic   = DDColorHF.from_pretrained("piddnad/ddcolor_artistic")

# python infer_hf.py --model_name ddcolor_artistic --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out_artistic --input_size 512

# python infer_hf.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512


/home/h/ddcolor312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Output path: out_train_ViCoW_1.0_0.5_freeze_g_pretrain_20000
100%|█████████████████████████████████████████| 382/382 [00:43<00:00,  8.70it/s]
/home/h/ddcolor312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Output path: out_train_ViCoW_1.0_0.5_freeze_g_pretrain_10000
100%|█████████████████████████████████████████| 382/382 [00:42<00:00,  9.09it/s]


## Validate

In [ ]:
import os
import cv2
import torch
import numpy as np
import lpips
from skimage import color
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ===================== SETUP =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn_vgg = lpips.LPIPS(net="vgg").to(device)

# ===================== METRICS =====================
def calculate_cf(img_gt, img_out):
    """
    Color Fidelity (CF) = PSNR trên kênh a,b (Lab)
    img_gt, img_out: BGR uint8 [0,255]
    return: CF (dB)
    """
    img_gt = cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)
    img_out = cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB)

    lab_gt = color.rgb2lab(img_gt)
    lab_out = color.rgb2lab(img_out)

    ab_gt = lab_gt[:, :, 1:3]
    ab_out = lab_out[:, :, 1:3]

    mse = np.mean((ab_gt - ab_out) ** 2)
    if mse == 0:
        return float("inf")

    return 10 * np.log10((255.0 ** 2) / mse)


def im2tensor(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = torch.from_numpy(image).permute(2, 0, 1).float()
    image = (image / 127.5) - 1.0
    return image.unsqueeze(0).to(device)

# ===================== EVAL =====================
def calculate_full_metrics(gt_dir, out_dir):
    gt_images = sorted(
        [f for f in os.listdir(gt_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    )

    m = {"psnr": [], "ssim": [], "cf": [], "lpips": []}

    print(f"--- Đang đánh giá {len(gt_images)} ảnh trên {device} ---")

    for filename in gt_images:
        path_gt = os.path.join(gt_dir, filename)
        path_out = os.path.join(out_dir, filename)
        if not os.path.exists(path_out):
            continue

        img_gt = cv2.imread(path_gt)
        img_out = cv2.imread(path_out)

        if img_gt.shape != img_out.shape:
            img_out = cv2.resize(img_out, (img_gt.shape[1], img_gt.shape[0]))

        m["psnr"].append(psnr(img_gt, img_out, data_range=255))
        m["ssim"].append(ssim(img_gt, img_out, channel_axis=2, data_range=255))

        cf_val = calculate_cf(img_gt, img_out)
        m["cf"].append(cf_val)

        with torch.no_grad():
            m["lpips"].append(
                loss_fn_vgg(im2tensor(img_gt), im2tensor(img_out)).item()
            )

        print(
            f"[{filename}] "
            f"PSNR: {m['psnr'][-1]:.2f} | "
            f"SSIM: {m['ssim'][-1]:.4f} | "
            f"CF: {cf_val:.2f} dB | "
            f"LPIPS: {m['lpips'][-1]:.4f}"
        )

    if m["psnr"]:
        print("\n" + "=" * 50)
        print("KẾT QUẢ TRUNG BÌNH TOÀN BỘ TẬP TEST")
        print(f"PSNR : {np.mean(m['psnr']):.2f} dB ↑")
        print(f"SSIM : {np.mean(m['ssim']):.4f} ↑")
        print(f"CF   : {np.mean(m['cf']):.2f} dB ↑")
        print(f"LPIPS: {np.mean(m['lpips']):.4f} ↓")
        print("=" * 50)
    else:
        print("Không có dữ liệu.")

# ===================== MAIN =====================
if __name__ == "__main__":
    folder_gt = "/home/h/DDColor/dataset/test"
    folder_out = "/home/h/DDColor/out_test_20000_1.2_0.5_freeze_ED_g_pretrain"
    calculate_full_metrics(folder_gt, folder_out)


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/h/ddcolor312/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth
--- Đang đánh giá 382 ảnh trên cuda ---
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0095.jpg] PSNR: 24.90 | SSIM: 0.9200 | CF: 27.34 dB | LPIPS: 0.3262
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0115.jpg] PSNR: 23.52 | SSIM: 0.9256 | CF: 26.55 dB | LPIPS: 0.3220
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0126.jpg] PSNR: 25.72 | SSIM: 0.9366 | CF: 29.01 dB | LPIPS: 0.2536
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0131.jpg] PSNR: 26.48 | SSIM: 0.9479 | CF: 30.03 dB | LPIPS: 0.2201
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0151.jpg] PSNR: 29.85 | SSIM: 0.9700 | CF: 32.48 dB | LPIPS: 0.1922
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0152.jpg] PSNR: 24.00 | SSIM: 0.9367 | CF: 26.70 dB | LPIPS: 0.2741
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0159.jpg] PSNR: 30.99 | SSIM: 0.9799 | CF: 33.74 dB | LPIPS: 0.1565
[VIDEO1_NhungNguoiVietLe

KeyboardInterrupt: 

## Demo Gradio

In [2]:
!pip install gradio gradio_imageslider timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 5.0 MB/s  0:00:04m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 1.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [gradio_imageslider]radio]]]


In [2]:
!python gradio_app_WIP.py

/home/h/ddcolor312/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Đang tải model...
Model đã sẵn sàng!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://00d636ef4b56b56b7d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Đang xử lý...
Xong!
Đang xử lý...
Xong!
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/home/h/ddcolor312/lib/python3.12/site-packages/uvicorn/protocols/http/h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/h